<a href="https://colab.research.google.com/github/Khewsingh/breast-cancer-hpo/blob/main/HPO_BreastCancer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hyperparameter Optimization — Breast Cancer Classification
**Algorithm:** Random Forest Classifier  
**Dataset:** Wisconsin Breast Cancer Dataset (569 samples, 30 features)  
**Goal:** Compare Manual HPO vs RandomizedSearchCV vs GridSearchCV  
**Tech Stack:** Python, Scikit-learn, Pandas, Seaborn

In [ ]:
# Libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

## 1. Data Loading & Exploration
Loading the dataset directly from GitHub. The dataset contains 30 features computed from breast mass cell nuclei images.  
**Target:** `diagnosis` — Malignant (M=1) or Benign (B=0).

In [ ]:
# Loading dataset from GitHub
dataset_url = "https://raw.githubusercontent.com/apogiatzis/breast-cancer-azure-ml-notebook/master/breast-cancer-data.csv"
dataset = pd.read_csv(dataset_url)
dataset.head()

In [ ]:
# Dataset Overview
print("Shape:", dataset.shape)
print("\nClass Distribution (count):")
print(dataset.diagnosis.value_counts())
print("\nClass Distribution (%):")
print(round(dataset.diagnosis.value_counts() / len(dataset) * 100, 2))

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution — dataset is slightly imbalanced (63% Benign, 37% Malignant)
sns.countplot(x='diagnosis', data=dataset, palette='Set2')
plt.title('Class Distribution (B=Benign, M=Malignant)')
plt.show()

In [ ]:
# Feature correlation heatmap
feature_cols_eda = [col for col in dataset.columns if col not in ['id', 'diagnosis', 'Unnamed: 32']]
plt.figure(figsize=(12, 8))
sns.heatmap(dataset[feature_cols_eda].corr(), cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.show()

In [ ]:
# Malignant tumors tend to have larger radius
plt.figure(figsize=(8, 5))
sns.boxplot(x='diagnosis', y='radius_mean', data=dataset)
plt.title("Radius Mean by Diagnosis")
plt.show()

In [ ]:
# Malignant tumors also have larger area
plt.figure(figsize=(8, 5))
sns.boxplot(x='diagnosis', y='area_mean', data=dataset)
plt.title("Area Mean by Diagnosis")
plt.show()

## 3. Data Preprocessing

In [ ]:
# Encoding target: Malignant (M) → 1, Benign (B) → 0
dataset['diagnosis'] = dataset['diagnosis'].map({'M': 1, 'B': 0})

In [ ]:
# Separating Features (X) and Target Variable (y)
# Dropping: 'id' (irrelevant), 'Unnamed: 32' (empty column)
# Using all 30 numeric features

feature_cols = [col for col in dataset.columns if col not in ['id', 'diagnosis', 'Unnamed: 32']]
X = dataset[feature_cols]
y = dataset['diagnosis'].values

print(f"Feature matrix shape: {X.shape}")   # Should be (569, 30)
print(f"Target vector shape: {y.shape}")    # Should be (569,)

## 4. Train-Test Split & Feature Scaling

In [ ]:
# Train-Test split (80% train, 20% test)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

In [ ]:
# Feature Scaling using StandardScaler
# fit_transform on train, only transform on test (to prevent data leakage)
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

## 5. Base Model (Random Forest — Default Params)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

result = accuracy_score(y_test, y_pred)
print("Accuracy of the Base RF model:", round(result * 100, 2), "%")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# Classification Report
cr = classification_report(y_test, y_pred)
print("\nClassification Report:")
print(cr)

## 6. Manual HPO
Manually tuning hyperparameters one at a time to understand their individual effect.

In [ ]:
# Tuning n_estimators — number of trees in the forest
n_estimators_list = [1, 2, 3, 10, 50, 100, 200]

for estim in n_estimators_list:
    model = RandomForestClassifier(n_estimators=estim, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    result = accuracy_score(y_test, y_pred)
    print(f"n_estimators: {estim:>4}  →  Accuracy: {round(result * 100, 2)}%")

In [ ]:
# Tuning min_samples_leaf (using best n_estimators=100)
leaf_size = [1, 2, 3, 4, 5, 10]

for i in leaf_size:
    model = RandomForestClassifier(n_estimators=100, min_samples_leaf=i, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    result = accuracy_score(y_test, y_pred)
    print(f"min_samples_leaf: {i:>2}  →  Accuracy: {round(result * 100, 2)}%")

## 7. RandomizedSearchCV
Instead of trying all combinations, RandomizedSearchCV samples randomly from the parameter space — faster and often finds near-optimal results.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

n_estimators = [int(x) for x in np.linspace(start=100, stop=1000, num=10)]
max_depth = [int(x) for x in np.linspace(start=10, stop=110, num=11)]
min_samples_leaf = [1, 2, 4, 10, 20, 50, 100]
min_samples_split = [2, 3, 4, 5, 8, 10, 20, 50, 100, 200]
bootstrap = [True, False]

random_grid = {
    'n_estimators': n_estimators,
    'max_depth': max_depth,
    'min_samples_split': min_samples_split,
    'bootstrap': bootstrap,
    'min_samples_leaf': min_samples_leaf
}

print("Total combinations:",
      len(n_estimators)*len(max_depth)*len(min_samples_leaf)*len(min_samples_split)*len(bootstrap))

In [ ]:
# Random search — 100 iterations, 3-fold CV
rf = RandomForestClassifier()
rf_random = RandomizedSearchCV(estimator=rf, param_distributions=random_grid,
                                n_iter=100, cv=3, n_jobs=-1, random_state=42)
rf_random.fit(X_train, y_train)
print("Best Params:", rf_random.best_params_)

In [ ]:
def evaluate(model, test_features, test_labels):
    predictions = model.predict(test_features)
    accuracy = accuracy_score(test_labels, predictions)
    print(f'Accuracy = {round(accuracy * 100, 2)}%')
    return accuracy

In [ ]:
# Base model (n_estimators=5) for comparison
base_model = RandomForestClassifier(n_estimators=5, random_state=42)
base_model.fit(X_train, y_train)
print("Base Model:")
base_accuracy = evaluate(base_model, X_test, y_test)

# Best RandomSearch model
best_random = rf_random.best_estimator_
print("\nBest RandomSearch Model:")
random_accuracy = evaluate(best_random, X_test, y_test)

print(f'\nImprovement: {round(100*(random_accuracy - base_accuracy)/base_accuracy, 2)}%')

## 8. GridSearchCV
Exhaustive search over a narrowed parameter grid (informed by RandomizedSearchCV results). Tries every combination — slower but thorough.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'bootstrap': [True],
    'max_depth': [80, 90, 100],
    'max_features': [2, 3],
    'min_samples_leaf': [1, 2],
    'min_samples_split': [8, 10],
    'n_estimators': [200, 300, 400]
}
# Total: 3×2×2×2×3 = 72 combinations × 3 folds = 216 fits

rf_gd = RandomForestClassifier()
grid_search = GridSearchCV(estimator=rf_gd, param_grid=param_grid,
                           cv=3, n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)
print("Best Params:", grid_search.best_params_)

In [ ]:
best_grid = grid_search.best_estimator_
print("Best GridSearch Model:")
grid_accuracy = evaluate(best_grid, X_test, y_test)

print(f'\nImprovement over base: {round(100*(grid_accuracy - base_accuracy)/base_accuracy, 2)}%')

## 9. Results Summary

| Method | Description | Accuracy |
|--------|-------------|----------|
| Base RF (default) | RandomForestClassifier() | ~93.86% |
| Manual HPO | Best n_estimators=100/200 | ~94.74% |
| RandomizedSearchCV | 100 random combinations, 3-fold CV | ~94% |
| GridSearchCV | 72 combinations, 3-fold CV | ~94% |

**Best Params (GridSearch):** `n_estimators=300, max_depth=80, max_features=2, min_samples_split=8`

**Key Takeaways:**
- More trees (n_estimators) generally improve accuracy up to a point
- GridSearchCV is most systematic but computationally expensive
- RandomizedSearchCV offers a great balance of speed and accuracy
- All tuned models outperform the default baseline